# Build CatBoost customer-product ranking data

## Goal

Create one ranking group for each customer purchase date. Every real paid product on that date is retained with `label = 1`, including first purchases. A small, reproducible set of real products receives `label = 0`:

- products the same customer bought earlier but did not buy on the current date;
- catalogue products the customer had never paid for before.

Every historical value is calculated from the same customer-product pair strictly before the group date. No customer, product history, or date is substituted. Dates are kept only as group metadata and are not model features.

## 1. Setup

In [ ]:
from pathlib import Path
import random

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
input_path = project_root / "data" / "interim" / "cleaned_purchases.csv"
training_output_path = project_root / "data" / "processed" / "catboost_training.csv"
group_metadata_path = project_root / "data" / "processed" / "catboost_group_metadata.csv"

random_seed = 42
historical_negatives_per_positive = 1
never_paid_negatives_per_positive = 1

## 2. Load, validate, and build the product catalogue

Source rows are sorted before history is built. The catalogue contains only real products and records when each product first appeared, so negative sampling cannot select a product before it existed in the data.

In [ ]:
identifier_dtypes = {
    "customer_id": "string",
    "product_id": "string",
    "product_category": "string",
    "business_line": "string",
}

receipts = pd.read_csv(
    input_path,
    dtype=identifier_dtypes,
    parse_dates=["purchase_date"],
)

required_columns = {
    "customer_id",
    "purchase_date",
    "product_id",
    "paid_quantity",
    "received_quantity",
    "product_category",
    "business_line",
}
missing_columns = sorted(required_columns - set(receipts.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

event_key_columns = ["customer_id", "purchase_date", "product_id"]
receipts = (
    receipts
    .drop(
        columns=["promotion_name", "gift_quantity", "bonus_amount"],
        errors="ignore",
    )
    .sort_values(event_key_columns)
    .reset_index(drop=True)
)

duplicate_count = int(receipts.duplicated(event_key_columns).sum())
non_positive_receipt_count = int(receipts["received_quantity"].le(0).sum())
assert duplicate_count == 0, f"Found {duplicate_count} duplicate event keys."
assert non_positive_receipt_count == 0, (
    f"Found {non_positive_receipt_count} non-positive receipts."
)

product_catalogue = (
    receipts
    .sort_values(["purchase_date", "product_id"])
    .groupby("product_id", sort=False)
    .agg(
        product_category=("product_category", "first"),
        business_line=("business_line", "first"),
        first_seen_date=("purchase_date", "min"),
    )
)
catalogue_products = sorted(product_catalogue.index.tolist())
product_first_seen = product_catalogue["first_seen_date"].to_dict()

print(f"Loaded {len(receipts):,} sorted receipt rows.")
print(f"Built a catalogue of {len(product_catalogue):,} real products.")

## 3. Create customer-date groups with truthful history

Each group belongs to exactly one customer and one purchase date. Positive candidates are the products actually paid for on that date. Historical negatives and never-paid negatives are sampled separately to control dataset size.

For a never-paid product, paid history is zero. Receipt history can still be nonzero when the customer previously received the product without paying for it. `average_days_between_customer_product_purchases` is the average interval between earlier paid purchases of this exact product by this exact customer. The category and business-line flags show whether this customer paid for any earlier product with the candidate's category or business line.

In [ ]:
def new_product_history():
    return {
        "paid_purchase_count": 0,
        "receipt_count": 0,
        "paid_quantity": 0.0,
        "received_quantity": 0.0,
        "last_paid_quantity": pd.NA,
        "last_received_quantity": pd.NA,
        "last_paid_purchase_date": None,
        "last_received_date": None,
        "paid_interval_days_sum": 0,
        "paid_interval_count": 0,
    }


random_generator = random.Random(random_seed)
candidate_rows = []
group_metadata_rows = []
next_group_id = 0

for customer_id, customer_events in receipts.groupby("customer_id", sort=False):
    product_history = {}
    customer_paid_products = set()
    customer_paid_categories = set()
    customer_paid_business_lines = set()

    for scoring_date, date_events in customer_events.groupby(
        "purchase_date",
        sort=False,
    ):
        paid_products = set(
            date_events.loc[
                date_events["paid_quantity"].gt(0),
                "product_id",
            ]
        )

        if paid_products:
            negative_limit = len(paid_products)
            historical_pool = sorted(customer_paid_products - paid_products)
            historical_sample_size = min(
                historical_negatives_per_positive * negative_limit,
                len(historical_pool),
            )
            historical_negatives = set(
                random_generator.sample(
                    historical_pool,
                    historical_sample_size,
                )
            )

            available_products = {
                product_id
                for product_id in catalogue_products
                if product_first_seen[product_id] <= scoring_date
            }
            never_paid_pool = sorted(
                available_products
                - customer_paid_products
                - paid_products
            )
            never_paid_sample_size = min(
                never_paid_negatives_per_positive * negative_limit,
                len(never_paid_pool),
            )
            never_paid_negatives = set(
                random_generator.sample(
                    never_paid_pool,
                    never_paid_sample_size,
                )
            )

            group_products = sorted(
                paid_products
                | historical_negatives
                | never_paid_negatives
            )

            for product_id in group_products:
                history = product_history.get(product_id)
                category = product_catalogue.at[
                    product_id,
                    "product_category",
                ]
                business_line = product_catalogue.at[
                    product_id,
                    "business_line",
                ]

                if history is None:
                    previous_paid_purchase_count = 0
                    previous_receipt_count = 0
                    previous_paid_quantity = 0.0
                    previous_received_quantity = 0.0
                    average_previous_received_quantity = pd.NA
                    last_paid_quantity = pd.NA
                    last_received_quantity = pd.NA
                    days_since_last_paid_purchase = pd.NA
                    days_since_last_received = pd.NA
                    average_days_between_customer_product_purchases = pd.NA
                else:
                    previous_paid_purchase_count = history[
                        "paid_purchase_count"
                    ]
                    previous_receipt_count = history["receipt_count"]
                    previous_paid_quantity = history["paid_quantity"]
                    previous_received_quantity = history[
                        "received_quantity"
                    ]
                    average_previous_received_quantity = (
                        history["received_quantity"]
                        / history["receipt_count"]
                    )
                    last_paid_quantity = history["last_paid_quantity"]
                    last_received_quantity = history[
                        "last_received_quantity"
                    ]
                    last_paid_date = history["last_paid_purchase_date"]
                    last_received_date = history["last_received_date"]
                    days_since_last_paid_purchase = (
                        (scoring_date - last_paid_date).days
                        if last_paid_date is not None
                        else pd.NA
                    )
                    days_since_last_received = (
                        scoring_date - last_received_date
                    ).days
                    average_days_between_customer_product_purchases = (
                        history["paid_interval_days_sum"]
                        / history["paid_interval_count"]
                        if history["paid_interval_count"] > 0
                        else pd.NA
                    )

                if product_id in paid_products:
                    candidate_type = "positive"
                elif product_id in historical_negatives:
                    candidate_type = "historical_negative"
                else:
                    candidate_type = "never_paid_negative"

                candidate_rows.append({
                    "group_id": next_group_id,
                    "customer_id": customer_id,
                    "product_id": product_id,
                    "product_category": category,
                    "business_line": business_line,
                    "previous_paid_purchase_count": (
                        previous_paid_purchase_count
                    ),
                    "previous_receipt_count": previous_receipt_count,
                    "previous_paid_quantity": previous_paid_quantity,
                    "previous_received_quantity": (
                        previous_received_quantity
                    ),
                    "average_previous_received_quantity": (
                        average_previous_received_quantity
                    ),
                    "last_paid_quantity": last_paid_quantity,
                    "last_received_quantity": last_received_quantity,
                    "days_since_last_paid_purchase": (
                        days_since_last_paid_purchase
                    ),
                    "days_since_last_received": days_since_last_received,
                    "average_days_between_customer_product_purchases": (
                        average_days_between_customer_product_purchases
                    ),
                    "has_purchased_category_before": int(
                        pd.notna(category)
                        and category in customer_paid_categories
                    ),
                    "has_purchased_business_line_before": int(
                        pd.notna(business_line)
                        and business_line in customer_paid_business_lines
                    ),
                    "label": int(product_id in paid_products),
                    "_candidate_type": candidate_type,
                })

            group_metadata_rows.append({
                "group_id": next_group_id,
                "customer_id": customer_id,
                "scoring_date": scoring_date,
            })
            next_group_id += 1

        # Current-date events become history only after candidates are built.
        for event in date_events.itertuples(index=False):
            history = product_history.setdefault(
                event.product_id,
                new_product_history(),
            )

            if event.paid_quantity > 0:
                previous_paid_date = history["last_paid_purchase_date"]
                if previous_paid_date is not None:
                    history["paid_interval_days_sum"] += (
                        scoring_date - previous_paid_date
                    ).days
                    history["paid_interval_count"] += 1
                history["paid_purchase_count"] += 1
                history["paid_quantity"] += event.paid_quantity
                history["last_paid_quantity"] = event.paid_quantity
                history["last_paid_purchase_date"] = scoring_date
                customer_paid_products.add(event.product_id)
                if pd.notna(event.product_category):
                    customer_paid_categories.add(event.product_category)
                if pd.notna(event.business_line):
                    customer_paid_business_lines.add(event.business_line)

            history["receipt_count"] += 1
            history["received_quantity"] += event.received_quantity
            history["last_received_quantity"] = event.received_quantity
            history["last_received_date"] = scoring_date

candidate_data = pd.DataFrame(candidate_rows)
group_metadata = pd.DataFrame(group_metadata_rows)

if candidate_data.empty:
    raise ValueError("No candidate rows were created.")

## 4. Validate and save

The checks prove that every source purchase survives as a positive, historical negatives have genuine prior paid history, never-paid negatives have zero prior paid history, and all groups contain both labels. Diagnostic candidate types are removed before saving.

In [ ]:
positive_mask = candidate_data["_candidate_type"].eq("positive")
historical_negative_mask = candidate_data["_candidate_type"].eq(
    "historical_negative"
)
never_paid_negative_mask = candidate_data["_candidate_type"].eq(
    "never_paid_negative"
)

expected_positive_rows = int(receipts["paid_quantity"].gt(0).sum())
expected_groups = int(
    receipts.loc[
        receipts["paid_quantity"].gt(0),
        ["customer_id", "purchase_date"],
    ].drop_duplicates().shape[0]
)

assert int(candidate_data["label"].sum()) == expected_positive_rows
assert int(positive_mask.sum()) == expected_positive_rows
assert candidate_data.loc[positive_mask, "label"].eq(1).all()
assert candidate_data.loc[
    historical_negative_mask,
    "label",
].eq(0).all()
assert candidate_data.loc[
    historical_negative_mask,
    "previous_paid_purchase_count",
].gt(0).all()
assert candidate_data.loc[
    never_paid_negative_mask,
    "label",
].eq(0).all()
assert candidate_data.loc[
    never_paid_negative_mask,
    "previous_paid_purchase_count",
].eq(0).all()

training_columns = [
    "group_id",
    "customer_id",
    "product_id",
    "product_category",
    "business_line",
    "previous_paid_purchase_count",
    "previous_receipt_count",
    "previous_paid_quantity",
    "previous_received_quantity",
    "average_previous_received_quantity",
    "last_paid_quantity",
    "last_received_quantity",
    "days_since_last_paid_purchase",
    "days_since_last_received",
    "average_days_between_customer_product_purchases",
    "has_purchased_category_before",
    "has_purchased_business_line_before",
    "label",
]
training_data = (
    candidate_data[training_columns]
    .sort_values(["group_id", "product_id"])
    .reset_index(drop=True)
)

forbidden_training_columns = {
    "purchase_date",
    "scoring_date",
    "paid_quantity",
    "received_quantity",
    "gift_quantity",
    "bonus_amount",
    "promotion_name",
    "month",
    "weekday",
    "_candidate_type",
}
assert forbidden_training_columns.isdisjoint(training_data.columns)
assert training_data["group_id"].is_monotonic_increasing
assert training_data.groupby("group_id")["customer_id"].nunique().eq(1).all()
assert training_data["previous_paid_purchase_count"].ge(0).all()
assert training_data["previous_receipt_count"].ge(0).all()
assert training_data["previous_paid_quantity"].ge(0).all()
assert training_data["previous_received_quantity"].ge(0).all()
assert training_data["has_purchased_category_before"].isin([0, 1]).all()
assert training_data[
    "has_purchased_business_line_before"
].isin([0, 1]).all()
assert (
    training_data["previous_received_quantity"]
    - training_data["previous_paid_quantity"]
).ge(-1e-9).all()
assert training_data["days_since_last_paid_purchase"].dropna().gt(0).all()
assert training_data["days_since_last_received"].dropna().gt(0).all()
assert training_data[
    "average_days_between_customer_product_purchases"
].dropna().gt(0).all()

group_label_summary = (
    training_data
    .groupby("group_id", sort=False)["label"]
    .agg(group_size="size", positive_count="sum")
)
assert len(group_label_summary) == expected_groups
assert group_label_summary["positive_count"].gt(0).all()
assert (
    group_label_summary["positive_count"]
    < group_label_summary["group_size"]
).all()
assert len(group_metadata) == expected_groups
assert group_metadata["group_id"].is_unique

training_output_path.parent.mkdir(parents=True, exist_ok=True)
training_data.to_csv(training_output_path, index=False)
group_metadata.to_csv(
    group_metadata_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved_training_data = pd.read_csv(
    training_output_path,
    dtype=identifier_dtypes,
)
saved_group_metadata = pd.read_csv(
    group_metadata_path,
    dtype={"customer_id": "string"},
    parse_dates=["scoring_date"],
)
assert list(saved_training_data.columns) == training_columns
assert len(saved_training_data) == len(training_data)
assert len(saved_group_metadata) == len(group_metadata)

first_purchase_positive_rows = int(
    (
        positive_mask
        & candidate_data["previous_paid_purchase_count"].eq(0)
    ).sum()
)
validation_summary = pd.Series({
    "training_rows": len(training_data),
    "ranking_groups": training_data["group_id"].nunique(),
    "positive_rows": expected_positive_rows,
    "first_purchase_positive_rows": first_purchase_positive_rows,
    "repeat_purchase_positive_rows": (
        expected_positive_rows - first_purchase_positive_rows
    ),
    "historical_negative_rows": int(historical_negative_mask.sum()),
    "never_paid_negative_rows": int(never_paid_negative_mask.sum()),
    "positive_rate": training_data["label"].mean(),
})

print(f"Saved CatBoost training data to {training_output_path}")
print(f"Saved group metadata to {group_metadata_path}")
validation_summary

## Next step

Use the separate group metadata to split complete customer-date groups chronologically. Pass `label` as the target and `group_id` as CatBoost ranking metadata; neither is an ordinary feature. All other saved columns are available to the model.